In [ ]:
!pip install streamlit pyngrok xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 79.7 MB/s eta 0:00:00


In [ ]:
# (optional) localtunnel needs node/npm -- install if you plan to use LocalTunnel
!apt-get update -qq
!apt-get install -y -qq nodejs npm


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package gyp.
(Reading database ... 126374 files and directories currently installed.)
Preparing to unpack .../000-gyp_0.1+20210831gitd6c5dd5-5_all.deb ...
Unpacking gyp (0.1+20210831gitd6c5dd5-5) ...
Selecting previously unselected package javascript-common.
Preparing to unpack .../001-javascript-common_11+nmu1_all.deb ...
Unpacking javascript-common (11+nmu1) ...
Selecting previously unselected package libjs-events.
Preparing to unpack .../002-libjs-events_3.3.0+~3.0.0-2_all.deb ...
Unpacking libjs-events (3.3.0+~3.0.0-2) ...
Selecting previously unselected package libjs-highlight.js.
Preparing to unpack .../003-libjs-highlight.js_9.18.5+dfsg1-1_all.deb ...
Unpacking libjs-highlight.js (9.18.5+dfsg1-1) ...
Selecting previously 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os

# --- Paths to model & encoders ---
model_path = "/content/drive/MyDrive/StreamModel/crime_model_final.joblib"
le_area_path = "/content/drive/MyDrive/StreamModel/le_area_final.joblib"
le_crime_path = "/content/drive/MyDrive/StreamModel/le_crime_final.joblib"
historical_summary_path = "/content/drive/MyDrive/StreamModel/historical_summary.csv"

# --- Load model & encoders ---
model = joblib.load(model_path)
le_area = joblib.load(le_area_path)
le_crime = joblib.load(le_crime_path)
historical_summary = pd.read_csv(historical_summary_path)

# --- Helper function for risk calculation ---
def calculate_predicted_probabilities_and_risk(area_name, years, quarters, predicted_df, historical_summary):
    filtered_predictions = predicted_df[
        (predicted_df['Area_Name'] == area_name) &
        (predicted_df['Year'].isin(years)) &
        (predicted_df['Quarter'].isin(quarters))
    ].copy()

    if filtered_predictions.empty:
        return pd.DataFrame()

    output_list = []

    for (y, q), group_df in filtered_predictions.groupby(['Year', 'Quarter']):
        total_predicted_count = group_df['Predicted_Crime_Count'].sum()

        if total_predicted_count == 0:
            group_df['Predicted_Probability'] = 0.0
            group_df['Risk_Level'] = 'Low'
        else:
            group_df['Predicted_Probability'] = group_df['Predicted_Crime_Count'] / total_predicted_count
            def map_probability_to_risk(prob):
                if prob < 0.01:
                    return 'Low'
                elif prob < 0.05:
                    return 'Medium'
                else:
                    return 'High'
            group_df['Risk_Level'] = group_df['Predicted_Probability'].apply(map_probability_to_risk)

        merged_df = pd.merge(
            group_df[['Area_Name', 'Crime_Type', 'Predicted_Crime_Count',
                      'Predicted_Probability', 'Risk_Level']],
            historical_summary,
            on=['Area_Name', 'Crime_Type'],
            how='left'
        )

        merged_df[['Crime_Count_StdDev', 'Min_Crime_Count', 'Max_Crime_Count']] = \
            merged_df[['Crime_Count_StdDev', 'Min_Crime_Count', 'Max_Crime_Count']].fillna(0)

        merged_df['Year'] = y
        merged_df['Quarter'] = q

        final_cols = ['Area_Name', 'Crime_Type', 'Year', 'Quarter',
                      'Predicted_Crime_Count', 'Predicted_Probability',
                      'Risk_Level', 'Crime_Count_StdDev', 'Min_Crime_Count',
                      'Max_Crime_Count']

        output_list.append(merged_df[final_cols])

    final_output_df = pd.concat(output_list).sort_values(
        by=['Year', 'Quarter', 'Predicted_Probability'],
        ascending=[True, True, False]
    )

    return final_output_df

# --- Streamlit UI ---
st.title("Crime Rate Prediction with Risk Assessment")

# Input widgets
area = st.selectbox("Select Police Force Area", le_area.classes_)
crime_types = st.multiselect("Select Crime Types", le_crime.classes_, default=list(le_crime.classes_))
start_year = st.number_input("Starting Year", min_value=2021, max_value=2030, value=2025)
start_quarter = st.selectbox("Starting Quarter", ["Q1", "Q2", "Q3", "Q4"])
time_period = st.slider("Number of Future Quarters to Predict", 1, 8, 4)

# --- Predict button ---
if st.button("Predict"):
    if not crime_types:
        st.warning("Please select at least one crime type.")
    else:
        # Build future periods
        future_periods = []
        quarter_order = ["Q1", "Q2", "Q3", "Q4"]
        start_q_index = quarter_order.index(start_quarter)

        for i in range(time_period):
            q_index = (start_q_index + i) % 4
            y = start_year + (start_q_index + i) // 4
            future_periods.append((y, quarter_order[q_index]))

        future_data = []
        for year, quarter in future_periods:
            for crime_type in crime_types:
                future_data.append({
                    "Year": year,
                    "Quarter": quarter,
                    "Area_Name": area,
                    "Crime_Type": crime_type
                })
        predict_df = pd.DataFrame(future_data)

        # Encode features
        predict_df['Area_Enc'] = le_area.transform(predict_df['Area_Name'])
        predict_df['Crime_Enc'] = le_crime.transform(predict_df['Crime_Type'])
        quarter_map = {'Q1': 0.0, 'Q2': 0.25, 'Q3': 0.5, 'Q4': 0.75}
        predict_df['Time_Index'] = predict_df['Year'] + predict_df['Quarter'].map(quarter_map)
        predict_df['Quarter_Enc'] = predict_df['Quarter'].map({'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4})
        predict_df['Quarter_sin'] = np.sin(2 * np.pi * predict_df['Quarter_Enc'] / 4)
        predict_df['Quarter_cos'] = np.cos(2 * np.pi * predict_df['Quarter_Enc'] / 4)
        predict_df['Area_Crime_Interaction'] = predict_df['Area_Enc'] * predict_df['Crime_Enc']

        # Make predictions
        feature_cols = ['Area_Enc', 'Crime_Enc', 'Year', 'Quarter_sin', 'Quarter_cos',
                        'Area_Crime_Interaction', 'Time_Index']
        predict_df['Predicted_Crime_Count'] = model.predict(predict_df[feature_cols])

        # Risk assessment for all future quarters
        years = [y for y, q in future_periods]
        quarters = [q for y, q in future_periods]

        st.subheader("Predicted Risk Assessment")
        risk_df = calculate_predicted_probabilities_and_risk(
            area, years, quarters, predict_df, historical_summary
        )

        risk_df = risk_df.reset_index(drop=True)
        risk_df = risk_df.loc[:, ~risk_df.columns.duplicated()]
        if risk_df.empty:
            st.warning("No predictions found for the selected parameters.")
        else:
            # Color-code risk levels
            def color_risk(val):
                if val == 'High':
                    return 'color: red'
                elif val == 'Medium':
                    return 'color: orange'
                elif val == 'Low':
                    return 'color: green'
                return ''
            st.dataframe(risk_df.style.applymap(color_risk, subset=['Risk_Level']))


Writing app.py


In [ ]:
!ngrok config add-authtoken 32KWiXc3fKxYoO5ejp3NQLiQO1M_6g3oc4X7PUw6TWNbwEmoS

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from pyngrok import ngrok

# Kill previous tunnels if any
ngrok.kill()

# Set your auth token
ngrok.set_auth_token("32KWiXc3fKxYoO5ejp3NQLiQO1M_6g3oc4X7PUw6TWNbwEmoS")

# Start Streamlit in background
get_ipython().system_raw("streamlit run app.py --server.port 8501 --server.headless true &")

# Create public URL
public_url = ngrok.connect(8501)
print("🌍 Streamlit app is live at:", public_url)

🌍 Streamlit app is live at: NgrokTunnel: "https://1b39cee6f0f4.ngrok-free.app" -> "http://localhost:8501"
